# 8. Example: Calling the OpenAI API

The OpenAI API is an API for calling language models from code.

This lesson connects ideas we have already practiced:

- URL
- method
- headers
- JSON body
- status code
- JSON response
- API key


## Before Running the OpenAI Call

To call the OpenAI API for real, you need an API key saved as `OPENAI_API_KEY` in your environment or local `.env` file.

This notebook is safe to run without a key. If no key is found, the OpenAI request cells will print a message and skip the network call.


In [1]:
import os
from pprint import pprint
import requests

TIMEOUT = 30
MODEL = "gpt-5.5"

## Load `.env` If Available


In [2]:
try:
    from dotenv import load_dotenv
except ImportError:
    print("python-dotenv is not installed, so Python will only use existing environment variables.")
else:
    load_dotenv()
    print("Loaded environment variables from .env if one exists.")

Loaded environment variables from .env if one exists.


## Check for the API Key

We only print a masked version of the key.


In [3]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY found:", api_key[:7] + "..." + api_key[-4:])
else:
    print("No OPENAI_API_KEY found. OpenAI API calls will be skipped.")

No OPENAI_API_KEY found. OpenAI API calls will be skipped.


## The HTTP Version

Under the hood, the OpenAI API is still an HTTP API.

This example uses `POST` because we are sending a prompt in the JSON body and asking the model to create a response.


In [4]:
def extract_response_text(response_json):
    if response_json.get("output_text"):
        return response_json["output_text"]

    pieces = []
    for item in response_json.get("output", []):
        for content in item.get("content", []):
            if content.get("type") in {"output_text", "text"}:
                pieces.append(content.get("text", ""))

    return "\n".join(piece for piece in pieces if piece)


def call_openai_with_requests(prompt):
    if not api_key:
        print("Skipping because OPENAI_API_KEY is not set.")
        return None

    url = "https://api.openai.com/v1/responses"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "input": prompt,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=TIMEOUT,
    )

    print(response.status_code)
    response.raise_for_status()
    return response.json()

openai_response = call_openai_with_requests(
    "Explain APIs in two short sentences for a beginner Python student."
)

Skipping because OPENAI_API_KEY is not set.


In [5]:
if openai_response:
    print(extract_response_text(openai_response))

## The SDK Version

The official Python SDK gives us a shorter way to make the same kind of request.

If your installed package is old, upgrade it in a terminal:

```bash
python3 -m pip install --upgrade openai
```


In [6]:
def call_openai_with_sdk(prompt):
    if not api_key:
        print("Skipping because OPENAI_API_KEY is not set.")
        return None

    try:
        from openai import OpenAI
    except ImportError:
        print("The installed openai package does not include the current OpenAI client.")
        print("Run: python3 -m pip install --upgrade openai")
        return None

    client = OpenAI(api_key=api_key)
    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )
    return response

sdk_response = call_openai_with_sdk(
    "Write one friendly sentence that encourages students learning APIs."
)

Skipping because OPENAI_API_KEY is not set.


In [7]:
if sdk_response:
    print(sdk_response.output_text)

## What Makes This an API Call?

The OpenAI API call has the same pieces as the public APIs we have used:

- **Endpoint**: `https://api.openai.com/v1/responses`
- **Method**: `POST`
- **Headers**: `Authorization` and `Content-Type`
- **Body**: JSON containing the model and input
- **Response**: JSON containing the model output


## Quick Review

The OpenAI API lets a Python program call a language model. That means an app, website, notebook, or agent can generate text from code instead of only using a chat interface.
